# 动态规划（编辑距离）

在本笔记本中，我们将使用动态规划来解决全局比对（编辑距离）问题，并了解如何利用回溯从 DP 表中重构出解。

## 环境配置说明：
### 如果在本地运行：
#### 首次配置
* 安装Anaconda：访问 https://www.anaconda.com/products/distribution
* 创建conda环境：`conda create -n cs131232 python=3.11`
  * 详见 https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html
* 激活环境：`conda activate cs131232`
* 安装pip：`conda install pip`
* 安装Jupyter：`conda install jupyter`
#### 每次启动
* 确保激活环境：`conda activate cs131232`
* 启动Jupyter：`jupyter notebook` 或 `jupyter lab`
* 运行以下单元格**并在需要时重启内核**

In [ ]:
# Install dependencies
!pip install -r requirements.txt --quiet

In [ ]:
import otter
assert (otter.__version__ >= "4.4.1"), "Please reinstall the requirements and restart your kernel."

grader = otter.Notebook("edit-dist.ipynb")
import numpy.random as random
import string
import random
import pylev
import tqdm
import time
from inputs1 import all_arrs, all_sols

rng_seed = 0

## Q1：编辑距离（全局比对）

编辑距离问题旨在找出将一个单词转换为另一个单词所需的最少插入、删除和替换字符的次数。该问题的一个重要应用是寻找两个字符串之间的全局比对，这在计算生物学中经常使用。

如教材所述：
> 衡量两个字符串之间距离的一种自然方式是看它们能够对齐（或匹配）的程度。从技术上讲，对齐只是将字符串上下排列的一种方式。例如，下面是 `SNOWY` 和 `SUNNY` 的两种可能对齐方式：
>
>    S−NOWY  |  −SNOW−Y <br />
>    SUNN−Y   &nbsp;|  SUN−−NY <br />
>    Cost: 3  &nbsp;&nbsp;&nbsp;&nbsp;| Cost: 5
>
> “−” 表示“间隙”（gap），可以在任一字符串中放置任意数量的间隙。对齐的代价是字母不同的列数。两个字符串之间的编辑距离就是它们最佳对齐的代价。

在本问题中，你将实现一个算法来计算两个字符串 `x` 和 `y` 之间的对齐。具体来说，你的算法应返回全局比对结果（如上所示），而不仅仅是表示编辑距离的整数值。

### 以下部分将引导你逐步实现该算法。  
**本部分包含不计分的多项选择题，用以测试你的理解。如果你愿意，可以直接跳到最后一个问题，那是唯一计分的问题。**

输入：
- `x`：字符串，长度为 `n`
- `y`：字符串，长度为 `m`

算法概要：
1. 按照课堂和教材中所述，计算 DP 子问题。
2. 利用第 1 步中记忆化的子问题，重构出最优全局比对。

第 1 步可以直接实现教材中给出的伪代码来完成。

第 2 步可以使用一种称为“回溯”的方法来计算，我们将在此逐步介绍。回忆一下，所有 DP 都有一个底层的 DAG，其中节点代表子问题。在该 DAG 上，DP 算法找出从 `(0,0)` 到 `(n,m)` 的最短路径。最短路径的长度就是我们的编辑距离，而路径中的边对应于全局比对。在我们的回溯算法中，我们从 `(n,m)` 开始，重构回到 `(0,0)` 的最短路径。由于我们从 `(n,m)` 出发，最终到达 `(0,0)`，我们是在逆向追踪第 1 步中的计算过程，因此称之为“回溯”。

__正确性检查（不计分）：__  
假设我们在字符串 `x` 和 `y` 上计算了 DP 矩阵，并且希望找到 `x` 的前 5 个字符与 `y` 的前 6 个字符之间的编辑距离。在底层的 DAG 上，这对应于从 `(0,0)` 到哪个节点的最短路径？请以包含两个整数的元组形式给出你的答案。

In [ ]:
node = (5, 6)

In [ ]:
grader.check("sc1 (optional)")

现在假设我们有一种重构这条最短路径的方法，我们需要将这条路径上的边转换为实际的对齐。

__正确性检查（不计分）：__ 假设我们的算法回溯到节点 `(i,j)`，并确定边 `(i-1,j)->(i,j)` 在这条最短路径上。到目前为止，算法根据从 `(i,j)` 到 `(n,m)` 的路径计算出了两个字符串 `x_align` 和 `y_align`。它们对应于子串 `x[i:n]` 和 `y[j:n]` 的对齐。给定这条新边，你应分别在 `x_align` 和 `y_align` 中添加哪些字符？请以整数列表的形式输入你的答案选项（即 `ans = [1]` 或 `ans = [1,2]`），其中每个整数代表以下选项之一：

1. 在 `x_align` 的开头添加一个空位（gap）
2. 在 `y_align` 的开头添加一个空位（gap）
3. 在 `x_align` 的开头添加 `x[i-1]`
4. 在 `y_align` 的开头添加 `y[j-1]`

*提示：由于每一步都需要保持 `len(x_align) == len(y_align)`，因此必须为两个字符串各添加一个字符。*

In [ ]:
ans = [2, 3]

In [ ]:
grader.check("sc2 (optional)")

__正确性检查（不计分）：__ 假设我们的算法回溯到节点 `(i,j)`，并确定边 `(i,j-1)->(i,j)` 在这条最短路径上。到目前为止，算法根据从 `(i,j)` 到 `(n,m)` 的路径计算出了两个字符串 `x_align` 和 `y_align`。它们对应于子串 `x[i:n]` 和 `y[j:n]` 的对齐。给定这条新边，你应分别在 `x_align` 和 `y_align` 中添加哪些字符？请以整数列表的形式输入你的答案选项（即 `ans = [1]` 或 `ans = [1,2]`），其中每个整数代表以下选项之一：

1. 在 `x_align` 的开头添加一个空位（gap）
2. 在 `y_align` 的开头添加一个空位（gap）
3. 在 `x_align` 的开头添加 `x[i-1]`
4. 在 `y_align` 的开头添加 `y[j-1]`

*提示：由于每一步都需要保持 `len(x_align) == len(y_align)`，因此必须为两个字符串各添加一个字符。*

In [ ]:
ans = [1, 4]

In [ ]:
grader.check("sc3 (optional)")

__正确性检查（不计分）：__ 假设我们的算法回溯到节点 `(i,j)`，并确定边 `(i-1,j-1)->(i,j)` 在这条最短路径上。到目前为止，算法根据从 `(i,j)` 到 `(n,m)` 的路径计算出了两个字符串 `x_align` 和 `y_align`。它们对应于子串 `x[i:n]` 和 `y[j:n]` 的对齐。给定这条新边，你应分别在 `x_align` 和 `y_align` 中添加哪些字符？请以整数列表的形式输入你的答案选项（即 `ans = [1]` 或 `ans = [1,2]`），其中每个整数代表以下四个选项之一：

1. 在 `x_align` 的开头添加一个空位（gap）
2. 在 `y_align` 的开头添加一个空位（gap）
3. 在 `x_align` 的开头添加 `x[i-1]`
4. 在 `y_align` 的开头添加 `y[j-1]`

*提示：由于每一步都需要保持 `len(x_align) == len(y_align)`，因此必须为两个字符串各添加一个字符。*

In [ ]:
ans = [3, 4]

In [ ]:
grader.check("sc4 (optional)")

由于我们已经知道如何将边转换为全局比对，现在我们要重构从 `(n,m)` 到 `(0,0)` 的实际路径。如果一条边 `(a,b)->(c,d)` 是最短路径的一部分，这意味着子问题 `(a,b)` 被用于计算 `(c,d)` 的解。对于编辑距离问题，我们知道子问题 `(i,j)` 是由 `(i-1,j)`、`(i,j-1)` 或 `(i-1,j-1)` 三者之一计算得到的。因此，如果节点 `(i,j)` 在最短路径中被访问，那么边 `(i-1,j)->(i,j)`、`(i,j-1)->(i,j)` 或 `(i-1,j-1)->(i,j)` 中的一条必定在最短路径中。

我们可以根据 DP 矩阵中的值来确定正确的边。回顾编辑距离的递推式：`dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + diff(i,j))`。这意味着 `dp[i - 1][j] + 1`、`dp[i][j - 1] + 1` 和 `dp[i - 1][j - 1] + diff(i,j)` 这三个值中至少有一个等于 `dp[i][j]`。如果某个值等于 `dp[i][j]`，那么对应的子问题就是可能的先前子问题；否则就不是。如果存在多个可能的先前子问题，你可以回溯到其中任意一个。

__正确性检查（不计分）：__ 假设你知道 `dp[i][j] = 5`，并且 DP 矩阵中有以下值。哪些子问题可能被用来计算 `dp[i][j]`？请以整数列表的形式输入你的答案选项（即 `ans = [1]` 或 `ans = [1,2]`）：

1. `dp[i-1][j] = 4`
2. `dp[i][j-1] = 5`
3. `dp[i-1][j-1] = 5, diff(i,j) = 0`

In [ ]:
ans = [1, 3]

In [ ]:
grader.check("sc5 (optional)")

__正确性检查（不计分）：__ 假设你知道 `dp[i][j] = 9`，并且 DP 矩阵中有以下值。哪些子问题可能被用来计算 `dp[i][j]`？请以整数列表的形式输入你的答案选项（即 `ans = [1]` 或 `ans = [1,2]`）：

1. `dp[i-1][j] = 9`
2. `dp[i][j-1] = 8`
3. `dp[i-1][j-1] = 9, diff(i,j) = 1`

In [ ]:
ans = [2]

In [ ]:
grader.check("sc6 (optional)")

按照这个逻辑，我们从 `(n,m)` 出发，反复找到前一个节点，直到到达 `(0,0)`。每一步回溯时，我们都根据所经过的边来更新比对。

## 编辑距离 (计分)

_分值：_ 50

In [ ]:
def edit_distance(x, y):
    """
    args:
        x:string = the first word.
        y:string = The second word.
    
    return:
        Tuple[String,String] = the optimum global alignment between x and y. The first string in the 
        tuple corresponds to x and the second to y. Use hypen's '-' to represent gaps in each string.
    """
    n = len(x)
    m = len(y)
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        row = dp[i]
        previous_row = dp[i - 1]
        xi = x[i - 1]
        for j in range(1, m + 1):
            substitute = previous_row[j - 1] + (xi != y[j - 1])
            insert = row[j - 1] + 1
            delete = previous_row[j] + 1
            row[j] = substitute if substitute < insert and substitute < delete else (insert if insert < delete else delete)

    x_aligned = []
    y_aligned = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (x[i - 1] != y[j - 1]):
            x_aligned.append(x[i - 1])
            y_aligned.append(y[j - 1])
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            x_aligned.append(x[i - 1])
            y_aligned.append("-")
            i -= 1
        else:
            x_aligned.append("-")
            y_aligned.append(y[j - 1])
            j -= 1

    x_aligned.reverse()
    y_aligned.reverse()
    return "".join(x_aligned), "".join(y_aligned)


__注意：你的解决方案不应花费过多时间运行。如果运行时间超过 60 秒，则说明速度太慢。参考实现平均需要 20 秒。__


In [ ]:
grader.check("edit_distance")

### 调试

为方便起见，此处粘贴了简化版的 otter 测试。在调试时，你可以随意添加任何你想要的打印语句或断言。

In [ ]:
start = time.time()
def check_word(original, aligned):
    ''' checks that the string `aligned` is obtained by only adding gaps to the string `original`'''

    assert len(aligned) >= len(original), "your function returned a string which is shorter than the input string!"
    i,j = 0,0
    while i < len(original) and j < len(aligned):
        while aligned[j] == '-' and j < len(aligned):
            j += 1
        assert original[i] == aligned[j], "your function returned a string which cannot be produced by only adding gaps!"
        i += 1
        j += 1
    while j < len(aligned):
        assert aligned[j] == '-', "your function returned a string which cannot be produced by only adding gaps!"
        j += 1

NUM_TRIALS = 200
LETTERS = string.ascii_lowercase
MIN_WORD_SIZE = 250
MAX_WORD_SIZE = 500

letters = string.ascii_lowercase
for i in tqdm.tqdm(range(NUM_TRIALS)):
    word1_size = random.randint(MIN_WORD_SIZE, MAX_WORD_SIZE)
    word2_size = random.randint(MIN_WORD_SIZE, MAX_WORD_SIZE)
    word1 = ''.join(random.choice(letters) for i in range(word1_size))
    word2 = ''.join(random.choice(letters) for i in range(word2_size))
    align1,align2 = edit_distance(word1, word2)

    assert len(align1) == len(align2), f"""a global alignment requires the two strings to be the same 
        length, your functions returns two strings of length {len(align1)} and {len(align2)}!"""

    check_word(word1, align1)
    check_word(word2, align2)

    dist = 0
    for a,b in zip(align1,align2):
        if a != b:
            dist += 1
    staff_distance = pylev.levenshtein(word1, word2)
    assert staff_distance == dist, f"""the inputs have an edit distance of {staff_distance}, but your
        strings have a distance of {dist}."""
finish = time.time()
assert finish - start < 60, "your solution timed out"

除了这些测试用例外，自己创建一些额外的测试用例可能会有所帮助。欢迎你在下面添加更多测试：

In [ ]:
# TODO: Your tests here!

## 提交

确保在运行下面的单元格之前，你已经按顺序运行了笔记本中的所有单元格，以便所有图像/图表都能在输出中显示。下面的单元格将为你生成一个用于提交的压缩文件。

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)